## 0.   Deep learning basics

In [7]:
import torch
import torch.nn as nn


def build_single_neuron() -> nn.Linear:
    """Le plus petit réseau possible : 1 entrée, 1 sortie, donc 2
    boutons (1 poids + 1 biais). Ne peut apprendre que des droites."""
    return nn.Linear(in_features=1, out_features=1)


def build_mlp(
    input_dim: int = 1, hidden_dim: int = 8, output_dim: int = 1
) -> nn.Sequential:
    """Perceptron multicouche : une couche cachée + ReLU + une couche de
    sortie. La ReLU entre les deux couches est INDISPENSABLE -- sans
    elle, l'architecture est mathématiquement équivalente à une seule
    couche linéaire (vérifié empiriquement, voir en-tête)."""
    return nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim),
    )


def count_parameters(model: nn.Module) -> int:
    """Compte le nombre total de "boutons" réglables du modèle."""
    return sum(p.numel() for p in model.parameters())


def train_regression(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    epochs: int = 1000,
    lr: float = 0.01,
    optimizer_name: str = "sgd",
    verbose_every: int | None = None,
) -> dict:
    """Boucle d'entraînement pour un problème de régression, avec les 4
    étapes du cycle explicitement séparées.

    optimizer_name : "sgd" (descente de gradient simple, pédagogique) ou
    "adam" (adaptatif, converge bien plus vite en pratique)."""
    criterion = nn.MSELoss()
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    historique = []
    for epoch in range(epochs):
        # ETAPE 3a : effacer les gradients de l'etape precedente
        optimizer.zero_grad()
        # ETAPE 1 : prediction
        predictions = model(X)
        # ETAPE 2 : mesure de l'erreur
        loss = criterion(predictions, y)
        # ETAPE 3b : calcul des gradients
        loss.backward()
        # ETAPE 4 : ajustement des boutons
        optimizer.step()

        historique.append(float(loss.item()))
        if verbose_every and (epoch + 1) % verbose_every == 0:
            print(f"epoch {epoch + 1:5}  erreur = {loss.item():.6f}")

    return {
        "final_loss": historique[-1],
        "initial_loss": historique[0],
        "history": historique,
        "n_parameters": count_parameters(model),
    }


def inspect_one_training_step(
    model: nn.Module, X: torch.Tensor, y: torch.Tensor, lr: float = 0.01
) -> dict:
    """Exécute UNE seule étape d'entraînement en retournant l'état
    détaillé de chaque phase -- utile pour comprendre le mécanisme
    plutôt que de le voir comme une boîte noire."""
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    avant = {nom: p.clone().detach() for nom, p in model.named_parameters()}

    optimizer.zero_grad()
    predictions = model(X)
    loss = criterion(predictions, y)
    loss.backward()

    gradients = {nom: p.grad.clone().detach() for nom, p in model.named_parameters()}

    optimizer.step()
    apres = {nom: p.clone().detach() for nom, p in model.named_parameters()}

    return {
        "predictions": predictions.detach(),
        "loss": float(loss.item()),
        "params_before": avant,
        "gradients": gradients,
        "params_after": apres,
    }

In [ ]:
# --- BLOC 1 : UN neurone, 2 boutons -- l'etat initial ---
import sys

import torch
import torch.nn as nn

torch.manual_seed(42)
neurone = build_single_neuron()

print("Boutons du neurone :", count_parameters(neurone))
print(f"  poids = {neurone.weight.item():.4f}")
print(f"  biais = {neurone.bias.item():.4f}")
print(
    f"  -> il calcule : sortie = {neurone.weight.item():.3f} * \
        entree + {neurone.bias.item():.3f}"
)
print("  (valeurs ALEATOIRES au depart)")

Boutons du neurone : 2
  poids = 0.7645
  biais = 0.8300
  -> il calcule : sortie = 0.765 * entree + 0.830
  (valeurs ALEATOIRES au depart)


In [ ]:
# --- BLOC 2 : UNE etape d'entrainement, decomposee ---
X = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor(
    [[2.0], [4.0], [6.0], [8.0]]
)  # on veut apprendre : sortie = 2 * entree

torch.manual_seed(42)
neurone = build_single_neuron()
etape = inspect_one_training_step(neurone, X, y, lr=0.01)

print("\nETAPE 1 - Predictions :", etape["predictions"].squeeze().numpy().round(3))
print(f"ETAPE 2 - Erreur mesuree : {etape['loss']:.4f}")
print(f"ETAPE 3 - Gradient du poids : {etape['gradients']['weight'].item():.4f}")
print(f"          Gradient du biais : {etape['gradients']['bias'].item():.4f}")
print(
    f"ETAPE 4 - Poids : {etape['params_before']['weight'].item():.4f} -> \
        {etape['params_after']['weight'].item():.4f}"
)

# Verification a la main de la formule : nouveau = ancien - (lr * gradient)
avant = etape["params_before"]["weight"].item()
grad = etape["gradients"]["weight"].item()
print(f"\nVerification : {avant:.4f} - (0.01 * {grad:.4f}) = {avant - 0.01 * grad:.4f}")


ETAPE 1 - Predictions : [1.595 2.359 3.124 3.888]
ETAPE 2 - Erreur mesuree : 7.0094
ETAPE 3 - Gradient du poids : -14.3819
          Gradient du biais : -4.5173
ETAPE 4 - Poids : 0.7645 -> 0.9084

Verification : 0.7645 - (0.01 * -14.3819) = 0.9084


In [9]:
# --- BLOC 3 : repeter le cycle -- la convergence ---
torch.manual_seed(42)
neurone = build_single_neuron()
resultats = train_regression(neurone, X, y, epochs=200, lr=0.01)

print(f"\nErreur initiale : {resultats['initial_loss']:.4f}")
print(f"Erreur finale   : {resultats['final_loss']:.6f}")
print(f"Poids appris    : {neurone.weight.item():.4f}  (objectif : 2.0)")
print(f"Biais appris    : {neurone.bias.item():.4f}  (objectif : 0.0)")


# --- BLOC 4 : LA limite du neurone unique -- un probleme NON lineaire ---
X2 = torch.tensor([[-2.0], [-1.0], [0.0], [1.0], [2.0]])
y2 = torch.tensor([[4.0], [1.0], [0.0], [1.0], [4.0]])  # sortie = entree^2

torch.manual_seed(42)
neurone = build_single_neuron()
r = train_regression(neurone, X2, y2, epochs=2000, lr=0.01)
print(f"\n1 neurone lineaire sur un probleme courbe -> erreur = {r['final_loss']:.4f}")
with torch.no_grad():
    print("Predictions :", neurone(X2).squeeze().numpy().round(2))
print("-> ECHEC : predit la meme valeur partout (impossible de courber une droite)")


Erreur initiale : 7.0094
Erreur finale   : 0.063587
Poids appris    : 1.7908  (objectif : 2.0)
Biais appris    : 0.6152  (objectif : 0.0)

1 neurone lineaire sur un probleme courbe -> erreur = 2.8000
Predictions : [2. 2. 2. 2. 2.]
-> ECHEC : predit la meme valeur partout (impossible de courber une droite)


In [ ]:
# --- BLOC 5 : la solution -- couche cachee + ReLU ---
torch.manual_seed(42)
mlp = build_mlp(1, 8, 1)
r = train_regression(mlp, X2, y2, epochs=3000, lr=0.05, optimizer_name="adam")
print(
    f"\nMLP avec ReLU ({count_parameters(mlp)} boutons) -> \
        erreur = {r['final_loss']:.6f}"
)
with torch.no_grad():
    print("Predictions :", mlp(X2).squeeze().numpy().round(3))
print("Attendu     :", y2.squeeze().numpy())


MLP avec ReLU (25 boutons) -> erreur = 0.000000
Predictions : [ 4.  1. -0.  1.  4.]
Attendu     : [4. 1. 0. 1. 4.]


In [ ]:
# --- BLOC 6 : preuve que c'est la ReLU qui compte, pas le nombre de neurones ---
torch.manual_seed(42)
sans_relu = nn.Sequential(nn.Linear(1, 8), nn.Linear(8, 1))  # MEME taille, PAS de ReLU
r = train_regression(sans_relu, X2, y2, epochs=3000, lr=0.05, optimizer_name="adam")
print(
    f"\nMEME architecture SANS ReLU ({count_parameters(sans_relu)} boutons) ->\
        erreur = {r['final_loss']:.4f}"
)
with torch.no_grad():
    print("Predictions :", sans_relu(X2).squeeze().numpy().round(2))
print("-> ECHEC IDENTIQUE au neurone unique : empiler des couches lineaires")
print("   sans activation revient mathematiquement a UNE couche lineaire.")


MEME architecture SANS ReLU (25 boutons) -> erreur = 2.8000
Predictions : [2. 2. 2. 2. 2.]
-> ECHEC IDENTIQUE au neurone unique : empiler des couches lineaires
   sans activation revient mathematiquement a UNE couche lineaire.


In [12]:
# --- BLOC 7 : a quoi ressemble ReLU ---
relu = nn.ReLU()
valeurs = torch.tensor([-3.0, -1.5, -0.2, 0.0, 0.2, 1.5, 3.0])
print(f"\n{'entree':>8} {'ReLU':>8}")
for e, s in zip(valeurs, relu(valeurs)):
    print(f"{e.item():>8.1f} {s.item():>8.1f}")
print("-> garde les positifs, ecrase les negatifs a zero")


  entree     ReLU
    -3.0      0.0
    -1.5      0.0
    -0.2      0.0
     0.0      0.0
     0.2      0.2
     1.5      1.5
     3.0      3.0
-> garde les positifs, ecrase les negatifs a zero


In [13]:
# --- BLOC 8 : effet du learning rate ---
print(f"\n{'lr':>8} {'poids appris':>14} {'erreur':>14}")
for lr in [0.001, 0.01, 0.1, 0.5]:
    torch.manual_seed(42)
    n = build_single_neuron()
    r = train_regression(n, X, y, epochs=100, lr=lr)
    print(f"{lr:>8} {n.weight.item():>14.4f} {r['final_loss']:>14.6f}")

# A retenir : lr trop petit = apprentissage trop lent ;
# lr trop grand = DIVERGENCE (erreur qui explose, poids a nan).


      lr   poids appris         erreur
   0.001         1.4715       0.439902
    0.01         1.7176       0.115825
     0.1         1.9818       0.000510
     0.5            nan            nan


In [14]:
# --- BLOC 9 : la divergence en direct ---
torch.manual_seed(42)
n = build_single_neuron()
opt = torch.optim.SGD(n.parameters(), lr=0.5)
perte = nn.MSELoss()
print("\nAvec lr=0.5 (trop grand) :")
for etape in range(5):
    opt.zero_grad()
    e = perte(n(X), y)
    e.backward()
    opt.step()
    print(
        f"  etape {etape + 1}: poids={n.weight.item():14.2f}  erreur={e.item():18.2f}"
    )
print("-> le pas est si grand qu'on SAUTE par-dessus le minimum a chaque fois")


Avec lr=0.5 (trop grand) :
  etape 1: poids=          7.96  erreur=              7.01
  etape 2: poids=        -44.43  erreur=            367.52
  etape 3: poids=        341.03  erreur=          19847.91
  etape 4: poids=      -2491.91  erreur=        1072317.00
  etape 5: poids=      18331.34  erreur=       57934056.00
-> le pas est si grand qu'on SAUTE par-dessus le minimum a chaque fois


In [15]:
# --- BLOC 10 : le piege d'oublier zero_grad() ---
torch.manual_seed(42)
n_ok = build_single_neuron()
opt = torch.optim.SGD(n_ok.parameters(), lr=0.01)
for _ in range(50):
    opt.zero_grad()
    perte(n_ok(X), y).backward()
    opt.step()

torch.manual_seed(42)
n_bug = build_single_neuron()
opt = torch.optim.SGD(n_bug.parameters(), lr=0.01)
for _ in range(50):
    perte(n_bug(X), y).backward()
    opt.step()  # zero_grad() OUBLIE !

print(f"\nAVEC zero_grad() : poids = {n_ok.weight.item():.4f}")
print(f"SANS zero_grad() : poids = {n_bug.weight.item():.4f}")
print("-> les gradients s'ACCUMULENT au lieu d'etre remplaces.")
print("   Bug silencieux : le code tourne sans erreur, mais l'entrainement est faux.")


# --- BLOC 11 : combien de neurones caches faut-il ? ---
print(f"\n{'neurones':>10} {'boutons':>10} {'erreur':>14}")
for taille in [1, 2, 4, 8, 32]:
    torch.manual_seed(42)
    m = build_mlp(1, taille, 1)
    r = train_regression(m, X2, y2, epochs=3000, lr=0.05, optimizer_name="adam")
    print(f"{taille:>10} {count_parameters(m):>10} {r['final_loss']:>14.6f}")


AVEC zero_grad() : poids = 1.6718
SANS zero_grad() : poids = 2.6647
-> les gradients s'ACCUMULENT au lieu d'etre remplaces.
   Bug silencieux : le code tourne sans erreur, mais l'entrainement est faux.

  neurones    boutons         erreur
         1          4       1.800000
         2          7       1.733333
         4         13       1.800000
         8         25       0.000000
        32         97       0.000000


In [1]:
import os

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

In [16]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# 6 mots dans le vocabulaire, chacun represente par 3 nombres
embedding = nn.Embedding(num_embeddings=6, embedding_dim=3)

print("La table complete (6 lignes, une par mot) :")
print(embedding.weight.detach().numpy().round(3))

print("\nSi je demande le mot n°2 :")
print(" ", embedding(torch.tensor(2)).detach().numpy().round(3))
print("C'est EXACTEMENT la ligne 2 de la table ci-dessus.")

print("\nNombre de boutons dans cette table :", embedding.weight.numel())
print("Ce sont des parametres entrainables :", embedding.weight.requires_grad)

La table complete (6 lignes, une par mot) :
[[ 1.927  1.487 -0.497]
 [ 0.44  -0.758  1.078]
 [ 0.801  1.681  0.356]
 [-0.687  0.61   1.335]
 [-0.232  0.042 -0.252]
 [ 0.86  -0.31  -0.396]]

Si je demande le mot n°2 :
  [0.801 1.681 0.356]
C'est EXACTEMENT la ligne 2 de la table ci-dessus.

Nombre de boutons dans cette table : 18
Ce sont des parametres entrainables : True


In [ ]:
from collections import Counter

import torch
import torch.nn as nn

from classical_ml.dataset_loader import load_movie_reviews

X_train, X_test, y_train, y_test = load_movie_reviews()

print("=== LE CORPUS REEL ===")
print("Nombre d'avis d'entrainement :", len(X_train))
print("Exemple d'avis :", X_train[0][:120].replace("\n", " "))
print()

# 1. Construire le vocabulaire A PARTIR du corpus
compteur = Counter()
for texte in X_train:
    compteur.update(texte.lower().split())

vocab = {"<pad>": 0, "<unk>": 1}
for mot, _ in compteur.most_common(5000):
    vocab[mot] = len(vocab)

print("=== LE VOCABULAIRE, construit depuis le corpus ===")
print("Taille :", len(vocab))
print("Quelques entrees mot -> identifiant :")
for mot in ["<pad>", "<unk>", "the", "movie", "good", "terrible"]:
    print(f"  '{mot}' -> id {vocab.get(mot, 'absent')}")
print()

# 2. Convertir un VRAI avis en identifiants
avis = "the movie was terrible"
ids = [vocab.get(m, 1) for m in avis.split()]
print("=== CONVERSION texte -> identifiants ===")
print(f"Texte  : '{avis}'")
print(f"IDs    : {ids}")
print()

# 3. La table d'embedding a maintenant une ligne par mot REEL du vocabulaire
embedding = nn.Embedding(len(vocab), 100, padding_idx=0)
print("=== LA TABLE D'EMBEDDING ===")
print("Forme :", tuple(embedding.weight.shape), "-> une ligne par mot du vocabulaire")
print(f"Nombre de boutons : {embedding.weight.numel():,}")
print()
print(f"Vecteur du mot 'terrible' (id {vocab['terrible']}), 8 premieres valeurs :")
print(" ", embedding(torch.tensor(vocab["terrible"])).detach().numpy()[:8].round(3))

In [17]:
import torch
import torch.nn as nn

torch.manual_seed(1)

# --- ATTRIBUTION 1 : mot -> identifiant (arbitraire) ---
vocab = {"bon": 0, "mauvais": 1, "terrible": 2, "excellent": 3}
print("ATTRIBUTION 1 (mot -> id), arbitraire :")
for mot, id_ in vocab.items():
    print(f"  '{mot}' -> id {id_}")

ATTRIBUTION 1 (mot -> id), arbitraire :
  'bon' -> id 0
  'mauvais' -> id 1
  'terrible' -> id 2
  'excellent' -> id 3


In [18]:
# --- ATTRIBUTION 2 : identifiant -> vecteur initial (aleatoire) ---
embedding = nn.Embedding(4, 2)
print("\nATTRIBUTION 2 (id -> vecteur), ALEATOIRE au depart :")
for mot, id_ in vocab.items():
    v = embedding.weight[id_].detach().numpy().round(3)
    print(f"  '{mot}' (id {id_}) -> {v}")
print("-> aucun rapport avec le SENS des mots pour l'instant")


ATTRIBUTION 2 (id -> vecteur), ALEATOIRE au depart :
  'bon' (id 0) -> [0.661 0.267]
  'mauvais' (id 1) -> [0.062 0.621]
  'terrible' (id 2) -> [-0.452 -0.166]
  'excellent' (id 3) -> [-1.523  0.382]
-> aucun rapport avec le SENS des mots pour l'instant


In [ ]:
# --- ATTRIBUTION 3 : le vecteur devient utile via l'entrainement ---
# On entraine : "bon" et "excellent" doivent donner une sortie POSITIVE,
# "mauvais" et "terrible" une sortie NEGATIVE
import numpy as np

classifieur = nn.Linear(2, 1)
opt = torch.optim.Adam(
    list(embedding.parameters()) + list(classifieur.parameters()), lr=0.1
)
perte = nn.BCEWithLogitsLoss()

mots_ids = torch.tensor([0, 1, 2, 3])  # bon, mauvais, terrible, excellent
cibles = torch.tensor([1.0, 0.0, 0.0, 1.0])  # positif, negatif, negatif, positif

for _ in range(300):
    opt.zero_grad()
    sortie = classifieur(embedding(mots_ids)).squeeze()
    perte(sortie, cibles).backward()
    opt.step()

print("\nATTRIBUTION 3 (apres entrainement) :")
for mot, id_ in vocab.items():
    v = embedding.weight[id_].detach().numpy().round(3)
    print(f"  '{mot}' (id {id_}) -> {v}")

# Similarite entre mots de meme polarite vs polarite opposee


def cos_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


v_bon = embedding.weight[0].detach().numpy()
v_excellent = embedding.weight[3].detach().numpy()
v_mauvais = embedding.weight[1].detach().numpy()

print(
    f"\nSimilarite 'bon' <-> 'excellent' (meme polarite) : \
        {cos_sim(v_bon, v_excellent):.3f}"
)
print(
    f"Similarite 'bon' <-> 'mauvais' (polarite opposee) : \
        {cos_sim(v_bon, v_mauvais):.3f}"
)


ATTRIBUTION 3 (apres entrainement) :
  'bon' (id 0) -> [ 2.962 -2.059]
  'mauvais' (id 1) -> [-2.261  2.933]
  'terrible' (id 2) -> [-2.882  2.252]
  'excellent' (id 3) -> [ 1.251 -2.457]

Similarite 'bon' <-> 'excellent' (meme polarite) : 0.881
Similarite 'bon' <-> 'mauvais' (polarite opposee) : -0.953


## 1.       text embedding

In [ ]:
# --- BLOC 1 : construire vocabulaire + encoder le vrai corpus ---
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
import torch

from classical_ml.data_loader import load_movie_reviews
from deep_learning.text_embedding import (
    AverageEmbeddingClassifier,
    build_glove_embedding_matrix,
    build_vocabulary,
    encode_texts,
    evaluate_classifier,
    train_classifier,
)

X_train, X_test, y_train, y_test = load_movie_reviews()

vocab = build_vocabulary(X_train, max_vocab_size=5000)
print("Taille du vocabulaire :", len(vocab))

X_tr = encode_texts(X_train, vocab, max_len=200)
X_te = encode_texts(X_test, vocab, max_len=200)
y_tr = torch.tensor([1.0 if la == "pos" else 0.0 for la in y_train])
y_te = torch.tensor([1.0 if la == "pos" else 0.0 for la in y_test])

print("Forme des donnees encodees :", X_tr.shape)

Taille du vocabulaire : 5002
Forme des donnees encodees : torch.Size([1600, 200])


In [24]:
# --- BLOC 2 : classifieur avec embedding ALEATOIRE ---
torch.manual_seed(42)
modele_aleatoire = AverageEmbeddingClassifier(vocab_size=len(vocab), embed_dim=50)
resultats = train_classifier(modele_aleatoire, X_tr, y_tr, epochs=10)
evaluation = evaluate_classifier(modele_aleatoire, X_te, y_te)
print(
    f"\nEmbedding aleatoire, 10 epoques -> accuracy test = {evaluation['accuracy']:.3f}"
)


Embedding aleatoire, 10 epoques -> accuracy test = 0.587


In [ ]:
# --- BLOC 3 : le meme, mais initialise avec GloVe (Phase 2) ---
from embeddings.glove import load_pretrained_glove

glove = load_pretrained_glove("glove-50")
matrice_glove = build_glove_embedding_matrix(vocab, glove, embed_dim=50)

mots_trouves = (matrice_glove.abs().sum(dim=1) != 0).sum().item()
print(f"\nMots du vocabulaire trouves dans GloVe : {mots_trouves} / {len(vocab)}")

torch.manual_seed(42)
modele_glove = AverageEmbeddingClassifier(
    vocab_size=len(vocab), embed_dim=50, pretrained_weights=matrice_glove
)
resultats_glove = train_classifier(modele_glove, X_tr, y_tr, epochs=10)
evaluation_glove = evaluate_classifier(modele_glove, X_te, y_te)
print(
    f"Embedding GloVe, 10 epoques      -> accuracy test = {
        evaluation_glove['accuracy']:.3f}"
)

print(
    f"\nGain de GloVe sur aleatoire : +{
        (evaluation_glove['accuracy'] - evaluation['accuracy']) * 100:.1f} points"
)


Mots du vocabulaire trouves dans GloVe : 4891 / 5002
Embedding GloVe, 10 epoques      -> accuracy test = 0.615

Gain de GloVe sur aleatoire : +2.8 points


## 2.       CNN

In [29]:
import torch

from deep_learning.CNN import TextCNN
from deep_learning.text_embedding import AverageEmbeddingClassifier

torch.manual_seed(42)

sequence_a = torch.tensor([[1, 2, 3, 0, 0]])  # ex: "not very good"
sequence_b = torch.tensor([[3, 2, 1, 0, 0]])  # memes tokens, ordre inverse

modele_moyenne = AverageEmbeddingClassifier(vocab_size=10, embed_dim=8)
modele_cnn = TextCNN(vocab_size=10, embed_dim=8, num_filters=4, kernel_sizes=(2,))

with torch.no_grad():
    print("Moyenne - sortie A :", modele_moyenne(sequence_a).item())
    print("Moyenne - sortie B :", modele_moyenne(sequence_b).item())
    print(
        "Identiques ?",
        torch.allclose(modele_moyenne(sequence_a), modele_moyenne(sequence_b)),
    )
    print()
    print("CNN - sortie A :", modele_cnn(sequence_a).item())
    print("CNN - sortie B :", modele_cnn(sequence_b).item())
    print(
        "Identiques ?", torch.allclose(modele_cnn(sequence_a), modele_cnn(sequence_b))
    )

Moyenne - sortie A : -1.0251495838165283
Moyenne - sortie B : -1.0251495838165283
Identiques ? True

CNN - sortie A : -0.8473215699195862
CNN - sortie B : -0.906749427318573
Identiques ? False


In [ ]:
# --- BLOC 2 : entrainement complet sur le vrai corpus, embedding aleatoire ---
from classical_ml.data_loader import load_movie_reviews
from deep_learning.text_embedding import (
    build_vocabulary,
    encode_texts,
    evaluate_classifier,
    train_classifier,
)

X_train, X_test, y_train, y_test = load_movie_reviews()
vocab = build_vocabulary(X_train, max_vocab_size=5000)
X_tr = encode_texts(X_train, vocab, max_len=100)
X_te = encode_texts(X_test, vocab, max_len=100)
y_tr = torch.tensor([1.0 if lo == "pos" else 0.0 for lo in y_train])
y_te = torch.tensor([1.0 if lo == "pos" else 0.0 for lo in y_test])

torch.manual_seed(42)
cnn_aleatoire = TextCNN(vocab_size=len(vocab), embed_dim=50)
print(
    f"\nNombre de parametres du CNN : {
        sum(p.numel() for p in cnn_aleatoire.parameters()):,}"
)

train_classifier(cnn_aleatoire, X_tr, y_tr, epochs=15)
resultats = evaluate_classifier(cnn_aleatoire, X_te, y_te)
print(
    f"CNN (embedding aleatoire, 15 epoques)     \
        -> accuracy test = {resultats['accuracy']:.3f}"
)


Nombre de parametres du CNN : 269,493
CNN (embedding aleatoire, 15 epoques) -> accuracy test = 0.585


In [ ]:
# --- BLOC 3 : le meme CNN, mais initialise avec GloVe ---
from deep_learning.text_embedding import build_glove_embedding_matrix
from embeddings.glove import load_pretrained_glove

glove = load_pretrained_glove("glove-50")
matrice_glove = build_glove_embedding_matrix(vocab, glove, embed_dim=50)

torch.manual_seed(42)
cnn_glove = TextCNN(
    vocab_size=len(vocab), embed_dim=50, pretrained_weights=matrice_glove
)
train_classifier(cnn_glove, X_tr, y_tr, epochs=15)
resultats_glove = evaluate_classifier(cnn_glove, X_te, y_te)
print(
    f"CNN (embedding GloVe, 15 epoques)     \
        -> accuracy test = {resultats_glove['accuracy']:.3f}"
)

CNN (embedding GloVe, 15 epoques)     -> accuracy test = 0.630


In [ ]:
# --- BLOC 4 : comparaison finale avec le classifieur moyenne ---
torch.manual_seed(42)
moyenne_glove = AverageEmbeddingClassifier(
    vocab_size=len(vocab), embed_dim=50, pretrained_weights=matrice_glove
)
train_classifier(moyenne_glove, X_tr, y_tr, epochs=15)
resultats_moyenne = evaluate_classifier(moyenne_glove, X_te, y_te)

print(f"\n{'Architecture':30} {'Accuracy':>10}")
print(f"{'Moyenne + GloVe':30} {resultats_moyenne['accuracy']:>10.3f}")
print(f"{'CNN + GloVe':30} {resultats_glove['accuracy']:>10.3f}")
print(f"{'CNN + aleatoire':30} {resultats['accuracy']:>10.3f}")
print(f"{'SVM lineaire (Phase 4, rappel)':30} {'0.835':>10}")


Architecture                     Accuracy
Moyenne + GloVe                     0.637
CNN + GloVe                         0.630
CNN + aleatoire                     0.585
SVM lineaire (Phase 4, rappel)      0.835


## 3.       RNN/LSTM/Bi-LSTM/GRU

In [33]:
# --- BLOC 1 : le probleme du RNN simple -- la memoire s'efface ---
import sys

sys.path.insert(0, "src")
import torch
import torch.nn as nn

torch.manual_seed(42)
rnn = nn.RNN(input_size=4, hidden_size=4, batch_first=True)

sequence_a = torch.randn(1, 15, 4)
sequence_b = sequence_a.clone()
sequence_b[0, 0] = torch.randn(4) * 10  # on change radicalement le PREMIER mot

_, etat_a = rnn(sequence_a)
_, etat_b = rnn(sequence_b)
print(
    "RNN simple - difference sur l'etat final apres avoir change le 1er mot :",
    (etat_a - etat_b).abs().mean().item(),
)
print("-> proche de 0 : le signal du debut est efface apres 15 etapes")

RNN simple - difference sur l'etat final apres avoir change le 1er mot : 4.76837158203125e-07
-> proche de 0 : le signal du debut est efface apres 15 etapes


In [34]:
# --- BLOC 2 : LSTM garde une trace mesurable ---
lstm = nn.LSTM(input_size=4, hidden_size=4, batch_first=True)
_, (etat_a_l, _) = lstm(sequence_a)
_, (etat_b_l, _) = lstm(sequence_b)
print(
    "\nLSTM - difference sur l'etat final :", (etat_a_l - etat_b_l).abs().mean().item()
)
print("-> non nul : le LSTM protege mieux l'information ancienne")


LSTM - difference sur l'etat final : 0.000121360644698143
-> non nul : le LSTM protege mieux l'information ancienne


In [35]:
# --- BLOC 3 : verifier manuellement les 4 portes du LSTM ---
torch.manual_seed(42)
cell = nn.LSTMCell(input_size=4, hidden_size=3)
h0 = torch.zeros(1, 3)
c0 = torch.zeros(1, 3)
mot1 = torch.randn(1, 4)

h1_pytorch, c1_pytorch = cell(mot1, (h0, c0))

poids_x = cell.weight_ih.chunk(4, dim=0)
poids_h = cell.weight_hh.chunk(4, dim=0)
biais_x = cell.bias_ih.chunk(4, dim=0)
biais_h = cell.bias_hh.chunk(4, dim=0)
W_i, W_f, W_g, W_o = poids_x
U_i, U_f, U_g, U_o = poids_h
b_i, b_f, b_g, b_o = [bx + bh for bx, bh in zip(biais_x, biais_h)]

porte_entree = torch.sigmoid(mot1 @ W_i.T + h0 @ U_i.T + b_i)
porte_oubli = torch.sigmoid(mot1 @ W_f.T + h0 @ U_f.T + b_f)
candidat = torch.tanh(mot1 @ W_g.T + h0 @ U_g.T + b_g)
porte_sortie = torch.sigmoid(mot1 @ W_o.T + h0 @ U_o.T + b_o)

c1_manuel = porte_oubli * c0 + porte_entree * candidat
h1_manuel = porte_sortie * torch.tanh(c1_manuel)

print(
    "\nCalcul manuel des 4 portes == calcul officiel PyTorch ?",
    torch.allclose(c1_manuel, c1_pytorch, atol=1e-5)
    and torch.allclose(h1_manuel, h1_pytorch, atol=1e-5),
)
print("Porte d'entree :", porte_entree.detach().numpy().round(3))
print("Porte d'oubli  :", porte_oubli.detach().numpy().round(3))
print("Porte de sortie:", porte_sortie.detach().numpy().round(3))


Calcul manuel des 4 portes == calcul officiel PyTorch ? True
Porte d'entree : [[0.449 0.604 0.174]]
Porte d'oubli  : [[0.07  0.108 0.835]]
Porte de sortie: [[0.381 0.279 0.155]]


In [ ]:
# --- BLOC 4 : entrainement complet des 3 architectures, embedding GloVe ---
from classical_ml.data_loader import load_movie_reviews
from deep_learning.RNN import TextBiLSTM, TextGRU, TextLSTM
from deep_learning.text_embedding import (
    build_glove_embedding_matrix,
    build_vocabulary,
    encode_texts,
    evaluate_classifier,
    train_classifier,
)
from embeddings.glove import load_pretrained_glove

X_train, X_test, y_train, y_test = load_movie_reviews()
vocab = build_vocabulary(X_train, max_vocab_size=5000)
X_tr = encode_texts(X_train, vocab, max_len=100)
X_te = encode_texts(X_test, vocab, max_len=100)
y_tr = torch.tensor([1.0 if la == "pos" else 0.0 for la in y_train])
y_te = torch.tensor([1.0 if la == "pos" else 0.0 for la in y_test])

glove = load_pretrained_glove("glove-50")
matrice_glove = build_glove_embedding_matrix(vocab, glove, embed_dim=50)

print(f"\n{'Architecture':20} {'Parametres':>12} {'Accuracy':>10}")
for nom, ModeleClasse in [
    ("LSTM", TextLSTM),
    ("Bi-LSTM", TextBiLSTM),
    ("GRU", TextGRU),
]:
    torch.manual_seed(42)
    modele = ModeleClasse(
        vocab_size=len(vocab), embed_dim=50, pretrained_weights=matrice_glove
    )
    nb_params = sum(p.numel() for p in modele.parameters())
    train_classifier(modele, X_tr, y_tr, epochs=15)
    resultats = evaluate_classifier(modele, X_te, y_te)
    print(f"{nom:20} {nb_params:>12,} {resultats['accuracy']:>10.3f}")

print(f"{'Rappel SVM (Phase 4)':20} {'~5000':>12} {'0.835':>10}")


Architecture           Parametres   Accuracy
LSTM                      260,885      0.582
Bi-LSTM                   271,669      0.600
GRU                       258,197      0.582
Rappel SVM (Phase 4)        ~5000      0.835
